In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, FloatType

from delta import *

import logging

logging.getLogger("py4j").setLevel(logging.DEBUG)

In [2]:
spark = (
    SparkSession
    .builder
    .master("local[*]")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

26/04/28 17:35:54 WARN Utils: Your hostname, joao-miguel resolves to a loopback address: 127.0.1.1; using 192.168.1.13 instead (on interface wlp0s20f3)
26/04/28 17:35:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/miguel/trabalho-eng-dados-delta-iceberg/.venv/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/miguel/.ivy2/cache
The jars for the packages stored in: /home/miguel/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-660ec452-a95d-464a-a649-d180a385ca94;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 154ms :: artifacts dl 7ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0 

In [3]:
spark

In [4]:
spark.sql( """ CREATE TABLE ar_condicionado_delta ( id INT, nm_modelo STRING, nm_marca STRING, valor DECIMAL(10,2), ano INT, potencia STRING ) USING delta """ )

DataFrame[]

In [5]:
spark.sql("select * from ar_condicionado_delta").show()

26/04/28 17:36:07 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---+---------+--------+-----+---+--------+
| id|nm_modelo|nm_marca|valor|ano|potencia|
+---+---------+--------+-----+---+--------+
+---+---------+--------+-----+---+--------+



In [6]:
from delta.tables import DeltaTable

carro = DeltaTable.forPath(spark, "./spark-warehouse/ar_condicionado_delta")


In [7]:
carro.history().show()

+-------+--------------------+------+--------+------------+--------------------+----+--------+---------+-----------+--------------+-------------+----------------+------------+--------------------+
|version|           timestamp|userId|userName|   operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics|userMetadata|          engineInfo|
+-------+--------------------+------+--------+------------+--------------------+----+--------+---------+-----------+--------------+-------------+----------------+------------+--------------------+
|      0|2026-04-28 17:36:...|  NULL|    NULL|CREATE TABLE|{partitionBy -> [...|NULL|    NULL|     NULL|       NULL|  Serializable|         true|              {}|        NULL|Apache-Spark/3.5....|
+-------+--------------------+------+--------+------------+--------------------+----+--------+---------+-----------+--------------+-------------+----------------+------------+--------------------+



In [8]:
spark.sql("""
    INSERT INTO ar_condicionado_delta 
    (id, nm_modelo, nm_marca, valor, ano, potencia)
    VALUES 
    (1, 'KOHI 09QC 1HV', 'KOMECO', 1798.90, 2023, '9000 BTU/h'),
    (2, 'B0DSLQP69S', 'SAMSUNG', 2089.10, 2024, '12000 BTUs'),
    (3, '42AGVCC30M5', 'SAMSUNG', 6590.00, 2025, '30000 BTUs')
""")

DataFrame[]

In [9]:
spark.sql("select * from ar_condicionado_delta").show()

+---+-------------+--------+-------+----+----------+
| id|    nm_modelo|nm_marca|  valor| ano|  potencia|
+---+-------------+--------+-------+----+----------+
|  1|KOHI 09QC 1HV|  KOMECO|1798.90|2023|9000 BTU/h|
|  3|  42AGVCC30M5| SAMSUNG|6590.00|2025|30000 BTUs|
|  2|   B0DSLQP69S| SAMSUNG|2089.10|2024|12000 BTUs|
+---+-------------+--------+-------+----+----------+



In [10]:
carro.history().show(truncate=False)

+-------+-----------------------+------+--------+------------+----------------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------+------------+-----------------------------------+
|version|timestamp              |userId|userName|operation   |operationParameters                                                                           |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                           |userMetadata|engineInfo                         |
+-------+-----------------------+------+--------+------------+----------------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------+------------+-----------------------------------+
|1  

In [12]:
spark.sql( """ alter table ar_condicionado_delta add column fl_inverter BOOLEAN, tensao STRING """ )

DataFrame[]

In [13]:
spark.sql(
    """
    select * from ar_condicionado_delta
    """
).show()

+---+-------------+--------+-------+----+----------+-----------+------+
| id|    nm_modelo|nm_marca|  valor| ano|  potencia|fl_inverter|tensao|
+---+-------------+--------+-------+----+----------+-----------+------+
|  1|KOHI 09QC 1HV|  KOMECO|1798.90|2023|9000 BTU/h|       NULL|  NULL|
|  3|  42AGVCC30M5| SAMSUNG|6590.00|2025|30000 BTUs|       NULL|  NULL|
|  2|   B0DSLQP69S| SAMSUNG|2089.10|2024|12000 BTUs|       NULL|  NULL|
+---+-------------+--------+-------+----+----------+-----------+------+



In [14]:
spark.sql( """ update ar_condicionado_delta set fl_inverter = true, tensao = '220v' where id in (2, 3) """ )

DataFrame[num_affected_rows: bigint]

In [15]:
spark.sql(
    """
    select * from ar_condicionado_delta
    """
).show()

+---+-------------+--------+-------+----+----------+-----------+------+
| id|    nm_modelo|nm_marca|  valor| ano|  potencia|fl_inverter|tensao|
+---+-------------+--------+-------+----+----------+-----------+------+
|  3|  42AGVCC30M5| SAMSUNG|6590.00|2025|30000 BTUs|       true|  220v|
|  2|   B0DSLQP69S| SAMSUNG|2089.10|2024|12000 BTUs|       true|  220v|
|  1|KOHI 09QC 1HV|  KOMECO|1798.90|2023|9000 BTU/h|       NULL|  NULL|
+---+-------------+--------+-------+----+----------+-----------+------+



In [17]:


DeltaTable.isDeltaTable(spark, "spark-warehouse/ar_condicionado_delta")

True

In [ ]:
spark.sql( """ update ar_condicionado_delta set fl_inverter = false, tensao = '170v' where id = 1 """ )

In [ ]:
spark.sql( """ delete from ar_condicionado_delta where id = 1 """ )

In [ ]:
spark.sql(
    """
    select * from ar_condicionado_delta
    """
).show()

In [19]:
spark.sql('describe HISTORY ar_condicionado_delta').show(truncate=False);

+-------+-----------------------+------+--------+------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+-----------------------------------+
|version|timestamp              |userId|userName|operation   |operationParameters                                                                                                                                                       |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                    